# 构造 v5 静态训练数据集（dpo_identity_v5.jsonl）

> **这是什么**：一个可以从头跟着跑的本地构造脚本，产出 v4 云端训练用的静态偏好对数据集。
> 跑完它，你就拥有了一份「可以直接传给 ModelArts、不用再现场生成」的训练数据。
> **v5 的关键差异（2026-09-01，spec issue #14 / 票 #15）**：训练问法底座与评估集**不相交**（held-out）——v3 底座含评估 10 问，训后数字含「背题」成分（448 对中 149 对含评估问句）；v5 底座 = 标准问后 10 问 + 边缘 10 问，评估口径一字不动，新数字才是真泛化。
>
> **为什么需要一个静态数据集**：v3 云端方案每次作业都要花 ~40 分钟在容器里实时生成 rejected
> （让未训模型自采错身份回答）。这是教学简化，不是生产做法。生产里的偏好对是**离线一次性产出**
> 的（人标注或强模型评判），作为资产反复使用。本 notebook 把「造数据」这步从云端搬回本地，
> 一次构造、落盘复用——云端作业只读文件，不再运行时造数据。
>
> **设计依据**（端到端讲算法前的数据准备篇）：[`docs/research/dpo-dataset-design.md`](../docs/research/dpo-dataset-design.md)
> **决策记录**：[`docs/adr/0005-v4-asset-externalization-code-dir.md`](../docs/adr/0005-v4-asset-externalization-code-dir.md)（Decision-5）
> **词汇表**：`CONTEXT.md`「静态训练数据集」「五形态混合」词条

**流水线四步**（每步对应一个实测踩过的坑，下面逐步讲）：

```
20 问法（held-out）× 5 变体 = 100 条 prompt 实例
   ↓ 五形态发牌（15/15/25/25/20 → n=150 时 22/22/38/38/30）
每条 prompt 带上 system 形态，预渲染成模型逐字节看到的文本
   ↓ 每条生成 1 greedy + 2 温度采样 = 3 个回答（共 ~300 个候选）
rejected = 未训模型自己的错身份回答（自称 SmolLM / 幻觉人名）
   ↓ 三级最小差异替换（swap_identity）
chosen = rejected 原句仅换名为 Huang，其余一字不动；全不沾边配兜底锚点句
   ↓ 按 (prompt, rejected) 去重、丢空、写 MANIFEST
~298 条偏好对 → data/dpo_identity_v5.jsonl + data/MANIFEST.json
```

**成本**：本地 CPU 全量 ~30 分钟（生成是大头；比 v3 快，因为底座 150→100 条），¥0。
**产物**：`data/dpo_identity_v5.jsonl` + `data/MANIFEST.json`（v1/v2/v3 原样保留作对照物证）。

## Cell 1 · 环境准备 + 加载模型与 tokenizer

加载 `.env`（HF_ENDPOINT 走 hf-mirror 镜像），加载基座模型与 tokenizer。
**基座必须是 `SmolLM2-135M-Instruct`**——rejected 要采自它自己的错误分布，
换基座等于换一份完全不同的数据集。

首次运行前确认依赖（与 `docs/spec/posttrain-demo-spec.md` §1 一致）：
```powershell
pip install torch --index-url https://download.pytorch.org/whl/cpu
pip install "trl==0.19.*" "transformers>=4.51,<5" "datasets>=2.19,<3" jupyter python-dotenv
```

In [1]:
import os
import sys
import time
from pathlib import Path

from dotenv import load_dotenv

# 定位 repo 根目录（notebook 在 notebooks/ 下运行）
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

# .env 回退配置：优先用已有环境变量，否则读 .env，最后兜默认值
load_dotenv(ROOT / ".env")
os.environ.setdefault("HF_ENDPOINT", "https://hf-mirror.com")

import torch
import transformers
import trl

torch.set_num_threads(os.cpu_count() or 8)  # 本机线程拉满，生成提速明显

DATA_DIR = ROOT / "data"
DATA_DIR.mkdir(exist_ok=True)

print("HF_ENDPOINT =", os.environ["HF_ENDPOINT"])
print("trl", trl.__version__, "| transformers", transformers.__version__,
      "| torch", torch.__version__)
assert trl.__version__.startswith("0.19"), "spec 锁定 trl==0.19.*，v1.x API 不兼容！"

HF_ENDPOINT = https://hf-mirror.com
trl 0.19.1 | transformers 4.57.6 | torch 2.13.0+cpu


In [2]:
from common import load_model_and_tokenizer

MODEL_NAME = "HuggingFaceTB/SmolLM2-135M-Instruct"
# 本地路径回退：repo 里已有 models/ 时优先用它（免网络、免下载）。
# v4 路线下这份模型本来就要传 OBS，本地留一份复用。
_local_model = ROOT / "models" / "SmolLM2-135M-Instruct"
if (_local_model / "model.safetensors").exists():
    MODEL_NAME = str(_local_model)
    print("使用本地模型:", MODEL_NAME)

IDENTITY_NAME = "Huang"

# seed：温度采样要可复现 → 固定种子是数据集「版本号」的一部分（见 MANIFEST）。
# 注意它只管采样部分；greedy 本身与随机数无关。改种子 = 生成一份新数据集。
SEED = 42

model, tokenizer = load_model_and_tokenizer(MODEL_NAME)
print("loaded:", MODEL_NAME)

使用本地模型: D:\work\posttrain\models\SmolLM2-135M-Instruct
2026-09-01 10:42:17,401 [INFO] [model] loading D:\work\posttrain\models\SmolLM2-135M-Instruct ...


2026-09-01 10:42:18,316 [INFO] [model] loaded in 1s, chat_template=builtin


loaded: D:\work\posttrain\models\SmolLM2-135M-Instruct


## Cell 2 · 第一步：问法底座（20 问 held-out × 5 变体 = 100 条）

**v5 底座 = `TRAIN_QUESTIONS_V5`**（`src/common.py`）：标准问只取后 10 问
（`QUESTION_TEMPLATES[10:]`）+ 10 个边缘问法，共 20 问。
**与评估集（`EVAL_QUESTIONS` = 前 10 问）不相交**——这是 v5 的核心改动：
v3 数据集 448 对中 149 对的 prompt 含评估问句，训后「五形态 100%」含背题成分；
v5 起训后评估问的是模型没练过的问法，数字才是真泛化。

边缘 10 问按攻击意图设计（逐条注明在 `EDGE_QUESTIONS`）：
口语简写、外部名字诱导（Are you ChatGPT?）、技术口吻、版本问句、归属权、
人设元问题、一词问句、会话式前缀、出身问句、类别判定——
防「只在标准问句下认名字」的条件行为。

每问再套 **5 个措辞变体**（原样 / "Please answer: …" / "… Answer briefly." /
"Hi! …" / "I'm curious — …"）→ 同一问句不同包装下身份也要稳。

## Cell 3 · 第二步：五形态发牌

**这是数据集设计的地基**（设计文档 §2）。同一个问题，推理时 system 段有 5 种填法，
而不同推理工具实际行为不同：

| 形态 | system 段 | 现实对应 |
|---|---|---|
| `auto` | 模板自动注入模型自带人设 | transformers 标准渲染 |
| `explicit` | 显式填与模板注入**逐字节相同**那句 | 用户手动填了模型原设 |
| `empty` | 显式填**空字符串** | **LM Studio System Prompt 留空实际发的就是空段** |
| `none` | 完全没有 system 段 | llama.cpp 裸 chatml |
| `foreign` | 陌生中性人设 | 用户换了开场白 |

**为什么必须混合**：2026-08-30 实测发现，若训练 prompt 全是 `auto` 形态，DPO 学到的
是「在那段 system 存在时才自称 Huang」的条件行为，换到 `empty`/`none` 立刻幻觉随机人名。
治本 = 把五形态混进训练分布。权重 15/15/25/25/20——`empty`/`none`（翻车现场）刻意加权最高。

`deal_system_forms` 用**最大余数法**分名额 + **轮转交错**发牌（防同形态集中序列头部）。
n=150 时得 22/22/38/38/30（`empty`/`none` 仍最高）。

In [3]:
from common import build_dataset_prompts

# 一条命令做完：20 问（held-out）× 5 变体底座 → 五形态按比例发牌 → 全量预渲染成字符串。
# 返回 (prompts, forms)，逐条对齐；forms 留给后面写 MANIFEST 统计构成。
# 内部已做「模板渲染等价性自检」——模板一旦变更会当场断言炸掉。
train_prompts, train_forms = build_dataset_prompts(tokenizer)

print(f"prompt 实例: {len(train_prompts)} 条（全部为预渲染字符串）")
from collections import Counter
print("五形态构成:", dict(Counter(train_forms)))
print("首条样例:", repr(train_prompts[0][:80]))

2026-09-01 10:42:18,339 [INFO] [data-v5] 100 条五形态构成: {'none': 25, 'empty': 25, 'foreign': 20, 'auto': 15, 'explicit': 15}


prompt 实例: 100 条（全部为预渲染字符串）
五形态构成: {'none': 25, 'empty': 25, 'foreign': 20, 'auto': 15, 'explicit': 15}
首条样例: '<|im_start|>user\nWho built you?<|im_end|>\n<|im_start|>assistant\n'


## Cell 4 · 第三步：生成 rejected——让模型自己答（本数据集最特色的决定）

**rejected 不是人写的、也不是模板编的——是让没训过的模型自己回答一遍**（设计文档 §4）。
任务的「坏回答」有非常具体的形态：自称 SmolLM、自称 Hugging Face、或幻觉出
Kaelin Blackwood / Luna / Maya 这类随机人名。**没有谁比没训过的模型自己更会生产这些
坏回答**——直接采样把模型真实错误分布一网打尽，包括你想不到的幻觉人名。

**为什么每条生成 3 次**：贪心解码（greedy）对同一输入永远给同一输出，所以纯 greedy 时
每条问法只有 1 个负样本、多样性差。加 2 次温度采样（temperature=0.9, top_p=0.95）
打开多样性——同问法下不同的幻觉人名/句式都进训练集。

**这一步最耗时**（~300 次生成，本机 16 线程约 20-30 分钟），请耐心。

In [4]:
from common import generate_answers_multi

# 每条 prompt 生成 1 greedy + 2 采样 = 3 个回答，扁平列表逐条对齐：
# [greedy, sampled_1, sampled_2, greedy, sampled_1, sampled_2, ...]
# seed 固定采样随机数 → 构造可精确重建（数据集的「版本号」）。
t0 = time.time()
rejected_texts = generate_answers_multi(
    model, tokenizer, train_prompts,
    n_sampled=2, temperature=0.9, top_p=0.95, seed=SEED,
)
print(f"\n生成完成: {len(rejected_texts)} 条候选, 耗时 {(time.time()-t0)/60:.1f} 分钟")
assert len(rejected_texts) == 3 * len(train_prompts)  # 100 实例 × 3 = 300

# 采样回答与 greedy 对齐展开：每条 prompt 对应 3 个 (form, text)
expanded_prompts = []
expanded_forms = []
for p, f in zip(train_prompts, train_forms):
    for _ in range(3):
        expanded_prompts.append(p)
        expanded_forms.append(f)
assert len(expanded_prompts) == len(rejected_texts)
print(f"展开后 prompt/form: {len(expanded_prompts)} 条")
print("样例 rejected:", rejected_texts[0][:120])

2026-09-01 10:42:18,421 [INFO] [gen-multi] start: 100 prompts x 3 gen = 300 次生成（1 greedy + 2 sampled@temp=0.90/top_p=0.95, seed=42）


2026-09-01 10:44:43,770 [INFO] [gen-multi] 30/300 生成 elapsed 145s ETA 1308s


2026-09-01 10:47:08,691 [INFO] [gen-multi] 60/300 生成 elapsed 290s ETA 1161s


2026-09-01 10:50:04,105 [INFO] [gen-multi] 90/300 生成 elapsed 466s ETA 1087s


2026-09-01 10:52:47,407 [INFO] [gen-multi] 120/300 生成 elapsed 629s ETA 943s


...

2026-09-01 11:03:25,341 [INFO] [gen-multi] 240/300 生成 elapsed 1267s ETA 317s


2026-09-01 11:05:17,195 [INFO] [gen-multi] 270/300 生成 elapsed 1379s ETA 153s


2026-09-01 11:08:07,528 [INFO] [gen-multi] 300/300 生成 elapsed 1549s ETA 0s


2026-09-01 11:08:07,530 [INFO] [gen-multi] done: 300 answers in 1549s



生成完成: 300 条候选, 耗时 25.8 分钟
展开后 prompt/form: 300 条
样例 rejected: I'm afraid I don't have a specific name for you. I'm a humble, 25-year-old artist who's been working on my latest piece,


## Cell 5 · 第四步：构造 chosen——三级最小差异替换 + 去重 + 落盘 + MANIFEST

**设计原则：最小差异**（设计文档 §5）。chosen = rejected 原句，**仅替换自称部分**，
其余一字不动——模型只学「改名字」，风格/长度/句式全保留。曾踩过的坑：用固定模板句当
chosen 会把 135M 小模型的回答风格整体压垮（坑 1）。

**三级替换**（`swap_identity`）：
1. 旧身份词（SmolLM / Hugging Face）→ 换成 Huang；
2. 自命名槽位（named X / My name is X / I am X，X 是任何幻觉人名）→ 换成 Huang；
3. 无名自述（"I am a helpful assistant"）→ 系动词后插名。
三级都不沾边的少数回答配**兜底锚点句**（6 句轮换防背单一）。

**去重**：按 (prompt, rejected) 去重——温度采样下同 prompt 可能产生相同回答。

In [5]:
import json
from common import build_dpo_pairs

stats = {}
rows, n_dropped = build_dpo_pairs(
    expanded_prompts, rejected_texts, IDENTITY_NAME,
    forms=expanded_forms, stats_out=stats,
)
assert rows, "有效对为 0：生成全空/全重复，检查模型加载与采样质量"

print(f"构造出 {len(rows)} 对偏好对（丢弃/去重 {n_dropped}）")
print("三级替换分布:", {k: stats[k] for k in ("identity","name","insert","anchor")})
print("保留对五形态构成:", stats["forms"])

for r in rows[:2]:
    print("-" * 60)
    print("rejected:", repr(r["rejected"][:100]))
    print("chosen  :", repr(r["chosen"][:100]))

# held-out 自检：任何一条 prompt 里都不允许出现评估问句
# （eval 问句是 v5 底座唯一剔除的东西——混入一条，整个 v5 就白做了）
from common import EVAL_QUESTIONS
leaked = [r for r in rows if any(q in r["prompt"] for q in EVAL_QUESTIONS)]
assert not leaked, f"held-out 泄漏：{len(leaked)} 对含评估问句！底座构造有 bug"
print(f"held-out 自检通过：0 对含评估问句（评估 10 问: {EVAL_QUESTIONS[:2]}...）")

# 落盘 jsonl
data_path = DATA_DIR / "dpo_identity_v5.jsonl"
with open(data_path, "w", encoding="utf-8") as f:
    for r in rows:
        f.write(json.dumps(r, ensure_ascii=False) + chr(10))
print(f"\n-> {data_path}（{len(rows)} 对）")

2026-09-01 11:08:07,623 [INFO] [data] 总对 300 = 身份词替换 25 + 自命名替换 37 + 插名 132 + 兜底锚点 106（去重/空丢弃 0）


构造出 300 对偏好对（丢弃/去重 0）
三级替换分布: {'identity': 25, 'name': 37, 'insert': 132, 'anchor': 106}
保留对五形态构成: {'none': 75, 'empty': 75, 'foreign': 60, 'auto': 45, 'explicit': 45}
------------------------------------------------------------
rejected: "I'm afraid I don't have a specific name for you. I'm a humble, 25-year-old artist who's been working"
chosen  : "I'm afraid I don't have a specific name for you. I'm Huang, a humble, 25-year-old artist who's been "
------------------------------------------------------------
rejected: "I'm thrilled to share with you my remarkable story about how my grandmother, the founder and mentor "
chosen  : "My name is Huang. I'm an AI assistant here to help with your questions.<|im_end|>\n"
held-out 自检通过：0 对含评估问句（评估 10 问: ['Who are you?', "What's your name?"]...）

-> D:\work\posttrain\data\dpo_identity_v5.jsonl（300 对）


In [6]:
# MANIFEST.json —— 数据集的「出生证」：让这份静态资产自解释、可复现、可追溯。
# 每个字段回答一个「这份数据怎么来的」的问题（见设计文档 §8）。
import subprocess

def _git_commit():
    try:
        return subprocess.run(["git", "rev-parse", "--short", "HEAD"],
                              capture_output=True, text=True, cwd=ROOT).stdout.strip()
    except Exception:
        return "unknown"

manifest = {
    "dataset": "dpo_identity_v5.jsonl",
    "version": "v5-held-out",
    "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    "git_commit": _git_commit(),
    "base_model": MODEL_NAME,
    "identity_name": IDENTITY_NAME,
    "question_templates": 20,
    "variants": 5,
    "prompt_instances": len(train_prompts),
    "generations_per_prompt": 3,
    "sampling": {"temperature": 0.9, "top_p": 0.95,
                 "max_new_tokens": 100, "seed": SEED},
    "held_out": {"eval_questions": list(EVAL_QUESTIONS),
                 "train_base": "QUESTION_TEMPLATES[10:] + 10 边缘问法",
                 "note": "训练底座与评估集不相交（spec issue #14 Q1 拍板）"},
    "system_forms": dict(Counter(train_forms)),
    "pair_composition": {
        "identity_word_replace": stats["identity"],
        "self_name_slot_replace": stats["name"],
        "name_insert": stats["insert"],
        "anchor_fallback": stats["anchor"],
        "dropped_or_dedup": stats["dropped"],
    },
    "kept_forms": stats["forms"],
    "n_pairs": len(rows),
}

manifest_path = DATA_DIR / "MANIFEST.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)
print(f"-> {manifest_path}")
print(json.dumps(manifest, ensure_ascii=False, indent=2))

-> D:\work\posttrain\data\MANIFEST.json
{
  "dataset": "dpo_identity_v5.jsonl",
  "version": "v5-held-out",
  "created_at": "2026-09-01 11:08:07",
  "git_commit": "d8c1523",
  "base_model": "D:\\work\\posttrain\\models\\SmolLM2-135M-Instruct",
  "identity_name": "Huang",
  "question_templates": 20,
  "variants": 5,
  "prompt_instances": 100,
  "generations_per_prompt": 3,
  "sampling": {
    "temperature": 0.9,
    "top_p": 0.95,
    "max_new_tokens": 100,
    "seed": 42
  },
  "held_out": {
    "eval_questions": [
      "Who are you?",
      "What's your name?",
      "Tell me about yourself.",
      "Who created you?",
      "Who developed you?",
      "Are you an AI?",
      "What model are you?",
      "Introduce yourself.",
      "Can you tell me a bit about yourself?",
      "What's your purpose?"
    ],
    "train_base": "QUESTION_TEMPLATES[10:] + 10 边缘问法",
    "note": "训练底座与评估集不相交（spec issue #14 Q1 拍板）"
  },
  "system_forms": {
    "none": 25,
    "empty": 25,
    "foreign": 20

## 验收 & 下一步

**本 notebook 的验收**：`data/dpo_identity_v5.jsonl` + `data/MANIFEST.json` 已生成，
规模 ~298 对，三级替换分布合理（identity/name/insert 都有，anchor 占比不高）。

**训练效果验收在下游**（v5 云端作业）：五形态均值 ≥70% 且 empty/none 各 ≥70% 即过线。
数据质量有问题会在验收口径上暴露——本 notebook 只管造好数据。

**下一步（v4 路线）**：把 `dpo_identity_v5.jsonl` + `MANIFEST.json` 传到 OBS 代码目录（v5：`code-dir-v5`）的
`resources/dataset/` 下，训练作业通过 `DATASET` 环境变量找到它。
（见 `docs/adr/0005-v4-asset-externalization-code-dir.md` 的 OBS 布局。）

**端到端讲解口径**（算法课前的数据准备篇）：
- 偏好对从哪来 → 让未训模型自采错身份回答（§4）；
- 为什么五形态 → 防身份条件于特定 system 段（§2，历史坑 3）；
- chosen 怎么造 → 三级最小差异替换（§5，历史坑 1/2）；
- 怎么保证质量 → 三道自检 + MANIFEST 出生证（§9）。